# 13 — Fundamentals Module Quickstart

Fundamentals snapshot, provider chain, Finnhub enrichment, and earnings proximity. See the [providers README](https://github.com/matteolongo/swing_screener/blob/main/src/swing_screener/fundamentals/providers/README.md) for full documentation.

In [ ]:
from __future__ import annotations
import os
import datetime as dt
from pathlib import Path

# Ensure CWD is the project root so instrument-master path resolution works.
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "pyproject.toml").exists():
        os.chdir(p)
        break

import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

from swing_screener.fundamentals.service import FundamentalsAnalysisService
from swing_screener.fundamentals.config import FundamentalsConfig
from swing_screener.fundamentals.earnings_proximity import fetch_next_earnings_days

## Fundamentals Snapshot

Fetch a fundamentals snapshot for AAPL. The service uses a provider chain (sec_edgar → yfinance) and returns a frozen dataclass (`FundamentalSnapshot`).

Finnhub enrichment requires `FINNHUB_API_KEY` — when unset, those fields (`analyst_recommendation_score`, `insider_net_shares_90d`, etc.) remain `None`.

In [ ]:
service = FundamentalsAnalysisService()
cfg = FundamentalsConfig()

try:
    snap = service.get_snapshot("AAPL", cfg=cfg)
    print(f"Sector: {snap.sector}")
    print(f"Market cap: {snap.market_cap}")
    print(f"Company: {snap.company_name}")
    print(f"Provider: {snap.provider}")
    print()
    # Show available top-level fields via to_dict()
    d = snap.to_dict()
    print(f"FundamentalSnapshot fields ({len(d)}):")
    for k, v in d.items():
        if not isinstance(v, (dict, list)):
            print(f"  {k}: {v}")
except Exception as exc:
    print(f"Snapshot fetch failed: {exc}")

## Earnings Proximity

Check how many days until the next earnings report for AAPL.

`fetch_next_earnings_days` first tries Finnhub; if no `FINNHUB_API_KEY` is set (or the key fails auth), it falls back to yfinance. A result of `None` means proximity could not be determined.

In [ ]:
finnhub_key = os.environ.get("FINNHUB_API_KEY") or None
if finnhub_key is None:
    print("Note: FINNHUB_API_KEY not set — earnings proximity will use yfinance fallback.")

prox = fetch_next_earnings_days(
    tickers=["AAPL"],
    finnhub_api_key=finnhub_key,
    asof_date=dt.date.today(),
    max_workers=1,
)
print(f"Earnings proximity for AAPL: {prox.get('AAPL')}  (None = unknown)")